# Claimify: extract factual claims

Extract claims from Markdown with **one H1 document title** and **H2 section subtitles**. Each sentence uses its entire enclosing section, document title, and section subtitle as context.

Run `uv sync` from the repository root and select the project's `.venv` kernel. The setup loads `.env` without overriding existing environment variables. Live extraction requires `TYPESAFE_API_KEY` and `OPENAI_API_KEY` and makes paid requests. Parsing and previewing the input are offline.

Jev makes semantic decisions; Luna filters, clarifies, and generates text. These claims are extracted assertions, not externally verified facts.

In [7]:
import html
import json
import os
from collections import Counter
from pathlib import Path
from time import perf_counter

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

from data_platform.digest.claimify import extract_claims, parse_document

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src/data_platform").is_dir()
)
load_dotenv(project_root / ".env", override=False)


def show_table(rows, columns):
    """Escape source/model text before displaying an HTML table."""
    if not rows:
        print("No rows.")
        return
    header = "".join(f"<th>{html.escape(label)}</th>" for label, key in columns)
    body = "".join(
        "<tr>" + "".join(
            f"<td style='white-space:pre-wrap'>{html.escape(str(row.get(key, '')))}</td>"
            for label, key in columns
        ) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

## Choose the Markdown file and models

Set `input_path` to your already segmented Markdown file. Relative paths resolve from the project root; absolute paths and `~` are supported. The default points to `.data/segmented/article-en-01.md`.

The file must contain one H1 document title and H2 section subtitles. Segmentation happens upstream; this notebook does not generate headings. Missing files and invalid document structure are reported before any model requests.

In [10]:
input_path = Path(".data/segmented/article-en-01.md")
question = None
# Both English and Vietnamese inputs are supported; output keeps the input language.
decision_model = "typesafe:jev-latest"
generation_model = "openai:gpt-6-luna"
max_concurrency = 4

input_path = Path(input_path).expanduser()
if not input_path.is_absolute():
    input_path = project_root / input_path
if not input_path.is_file():
    raise FileNotFoundError(f"Markdown file not found: {input_path}. Set input_path to an existing segmented Markdown file.")

markdown_doc = input_path.read_text(encoding="utf-8")
document = parse_document(markdown_doc)
print(f"Input: {input_path}")
print(f"{document.title}: {len(document.sections)} sections")
display(Markdown(markdown_doc))

Input: /Users/chuminhquan/projects/data-platform/.data/segmented/article-en-01.md
VINGROUP AND LEGENDS GLOBAL ANNOUNCE STRATEGIC PARTNERSHIP FOR VINFAST - TRONG DONG STADIUM: 7 sections


# VINGROUP AND LEGENDS GLOBAL ANNOUNCE STRATEGIC PARTNERSHIP FOR VINFAST - TRONG DONG STADIUM

## Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.

Hanoi, Vietnam, Vingroup - Vietnam’s largest private conglomerate, and Legends Global, the premier partner to the world’s greatest live events, venues and brands, have announced a strategic partnership for the management and operation of VinFast – Trong Dong Stadium. The agreement marks a significant milestone ahead of the landmark venue’s scheduled opening on July 31, 2027.

Under the agreement, Vingroup and Legends Global will work together to establish VinFast – Trong Dong Stadium as a premier global destination for world-class sporting, cultural, and entertainment events. In addition to helping attract major international events to the stadium, Legends Global will collaborate with Vingroup to oversee venue and facility management, vendor coordination, and on-site service operations, ensuring that every aspect meets the highest international standards of professionalism and excellence.

Drawing on its extensive experience managing more than 450 venues worldwide, hosting over 20,000 major events, and welcoming more than 165 million guests annually, Legends Global will also play a key role in developing the expertise and capabilities of Vingroup’s venue management professionals in Vietnam.

## Mr\. Nguyen Viet Quang\, Vice Chairman and CEO of Vingroup\, described VinFast – Trong Dong Stadium as more than a sporting venue\.

Commenting on the strategic partnership, Mr. Nguyen Viet Quang, Vice Chairman and CEO of Vingroup, said: “VinFast - Trong Dong Stadium is far more than a sporting venue. With its world-leading scale, contemporary architecture inspired by Vietnamese cultural heritage, automated retractable roof, and playing surface designed to meet FIFA standards, we envision working alongside Legends Global to transform the stadium into a global gathering place for the finest sporting, cultural, and entertainment events.”

## Mr\. Dan Levy\, CEO of Legends Global\, described VinFast – Trong Dong Stadium as an extraordinary project for Vietnam\.

Mr. Dan Levy, CEO of Legends Global, commented: “VinFast – Trong Dong Stadium is an extraordinary project for Vietnam, one that will help position Hanoi at the forefront of the global live events industry. Legends Global is honored to partner with Vingroup on this landmark development. We look forward to sharing our expertise in venue and event management to help bring Vingroup’s ambition and vision for the world’s largest stadium to life.”

## Mr\. Peter Loxton\, Chief Operating Officer for Asia Pacific and the Middle East at Legends Global\, said the stadium’s first major event would deliver an experience matching its extraordinary scale\.

Mr. Peter Loxton, Chief Operating Officer for Asia Pacific and the Middle East at Legends Global, affirmed: “We are confident that, from the very first major event, the experience for fans, athletes, performers, event organizers, and partners will match the extraordinary scale of the stadium itself. Our long-term exclusive partnership with Vingroup allows Legends Global to combine global expertise with local talent to strengthen Vietnam’s venue management capabilities.”

## VinFast – Trong Dong Stadium features a distinctive\, one\-of\-a\-kind design

VinFast – Trong Dong Stadium features a distinctive, one-of-a-kind design befitting the worlds most iconic destination landmarks. Inspired by the Dong Son bronze drum and the legendary Lac bird, enduring symbols of Vietnams thousand-year cultural heritage, its architecture blends traditional imagery with a bold, contemporary aesthetic that conveys strength and national pride. The stadium is also set to establish new benchmarks in scale and technological sophistication, with a planned capacity of 135,000 seats, the largest in the world, alongside integrated real-time security management and crowd-flow monitoring systems designed to support safe, efficient, and sustainable operations.

Surrounding the stadium is an extensive ecosystem of training grounds and large-scale sports facilities within the Hung Vuong International Sports Complex. The development will also feature a comprehensive range of amenities, including hotels, shopping centers, hospitals, parks, cultural and culinary villages, and theaters, offering a comfortable and convenient experience for athletes and fans from Vietnam and around the world.

Through the strategic partnership between Vingroup and Legends Global, VinFast – Trong Dong Stadium and the wider Hung Vuong International Sports Complex are expected to emerge as a new international destination on the global map of premier sporting and entertainment events. The development is anticipated to attract international visitors, stimulate the sports economy, and elevate Hanois profile on the world stage.

The partnership also lays the foundation for a long-term collaboration between the two organizations, with Legends Global providing exclusive stadium, arena, and live event services to Vingroup in Vietnam. Mr. Paul Sergeant OBE, a seasoned venue industry veteran and stadium expert at Legends Global, has already arrived in Hanoi to personally oversee technical preparations and pre-opening activities. Meanwhile, recruitment for the stadiums operations team is actively underway, establishing the groundwork for the venue to commence operations in accordance with international standards./.

## ABOUT VINGROUP

Vingroup is Vietnams largest private conglomerate and was ranked 57th on TIME magazines list of the Best Companies in Asia-Pacific 2026. Throughout more than three decades of development, Vingroup has consistently pushed new boundaries, created new engines of growth, and progressively expanded the reach of Vietnams business ecosystem onto the global stage. Today, companies within the Vingroup ecosystem operate in 12 countries, providing employment for more than 400,000 people across six core pillars: Industrials & Technology, Real Estate & Services, Infrastructure, Green Energy, Social Enterprises, and Culture & Sports.

For more information, please visit: https://vingroup.net.

## ABOUT LEGENDS GLOBAL

Legends Global is a premier international partner to the worlds most prestigious live events, venues, and brands. The company provides comprehensive, integrated solutions spanning feasibility studies, consulting, project development, sales, partnership development, venue management, premium hospitality, merchandise and content commercialization, and event production. Leveraging the strength of its global network of more than 450 venues, 20,000 events, and 165 million guests annually, Legends Global works with its partners to enhance commercial performance and deliver exceptional guest experiences.

Legends Global attracts events to, partners with, operates, and manages a global network of venues, including:

FIFA World Cup Host Stadiums: Reliant Stadium, Houston; AT&T Stadium, Dallas; SoFi Stadium, Los Angeles; Levi’s Stadium, Santa Clara; Rose Bowl, Los Angeles; Santiago Bernabéu, Madrid; Spotify Camp Nou, Barcelona.

Iconic National Stadiums: Emirates Stadium, London; Stamford Bridge, London; Suncorp Stadium, Brisbane; Kai Tak Stadium, Hong Kong; Optus Stadium, Perth; Yankee Stadium, New York; Caesars Superdome, New Orleans.

For more information, please visit: www.LegendsGlobal.com


## Inspect sentence context and provenance

Offsets are half-open Python character positions into the original Markdown. Headings are context, not claim targets. Code, tables, and HTML blocks are excluded from sentence extraction but remain in the section context.

In [11]:
sentence_rows = [
    {
        "index": sentence.index,
        "subtitle": section.subtitle,
        "text": sentence.text,
        "span": f"[{sentence.start}, {sentence.end})",
    }
    for section in document.sections for sentence in section.sentences
]
show_table(sentence_rows, [("Sentence", "index"), ("Subtitle", "subtitle"), ("Source text", "text"), ("Span", "span")])

section_index = 0
if document.sections:
    section = document.sections[section_index]
    print(json.dumps({
        "document_title": document.title,
        "section_subtitle": section.subtitle,
        "section_content": section.text,
        "excluded_spans": [span.model_dump() for span in section.excluded_spans],
    }, ensure_ascii=False, indent=2))

Sentence,Subtitle,Source text,Span
0,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"Hanoi, Vietnam, Vingroup - Vietnam’s largest private conglomerate, and Legends Global, the premier partner to the world’s greatest live events, venues and brands, have announced a strategic partnership for the management and operation of VinFast – Trong Dong Stadium.","[214, 481)"
1,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"The agreement marks a significant milestone ahead of the landmark venue’s scheduled opening on July 31, 2027.","[481, 591)"
2,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"Under the agreement, Vingroup and Legends Global will work together to establish VinFast – Trong Dong Stadium as a premier global destination for world-class sporting, cultural, and entertainment events.","[593, 796)"
3,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"In addition to helping attract major international events to the stadium, Legends Global will collaborate with Vingroup to oversee venue and facility management, vendor coordination, and on-site service operations, ensuring that every aspect meets the highest international standards of professionalism and excellence.","[796, 1115)"
4,Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\.,"Drawing on its extensive experience managing more than 450 venues worldwide, hosting over 20,000 major events, and welcoming more than 165 million guests annually, Legends Global will also play a key role in developing the expertise and capabilities of Vingroup’s venue management professionals in Vietnam.","[1117, 1423)"
5,"Mr\. Nguyen Viet Quang\, Vice Chairman and CEO of Vingroup\, described VinFast – Trong Dong Stadium as more than a sporting venue\.","Commenting on the strategic partnership, Mr. Nguyen Viet Quang, Vice Chairman and CEO of Vingroup, said: “VinFast - Trong Dong Stadium is far more than a sporting venue.","[1561, 1730)"
6,"Mr\. Nguyen Viet Quang\, Vice Chairman and CEO of Vingroup\, described VinFast – Trong Dong Stadium as more than a sporting venue\.","With its world-leading scale, contemporary architecture inspired by Vietnamese cultural heritage, automated retractable roof, and playing surface designed to meet FIFA standards, we envision working alongside Legends Global to transform the stadium into a global gathering place for the finest sporting, cultural, and entertainment events.”","[1730, 2071)"
7,"Mr\. Dan Levy\, CEO of Legends Global\, described VinFast – Trong Dong Stadium as an extraordinary project for Vietnam\.","Mr. Dan Levy, CEO of Legends Global, commented: “VinFast – Trong Dong Stadium is an extraordinary project for Vietnam, one that will help position Hanoi at the forefront of the global live events industry.","[2198, 2403)"
8,"Mr\. Dan Levy\, CEO of Legends Global\, described VinFast – Trong Dong Stadium as an extraordinary project for Vietnam\.",Legends Global is honored to partner with Vingroup on this landmark development.,"[2403, 2484)"
9,"Mr\. Dan Levy\, CEO of Legends Global\, described VinFast – Trong Dong Stadium as an extraordinary project for Vietnam\.",We look forward to sharing our expertise in venue and event management to help bring Vingroup’s ambition and vision for the world’s largest stadium to life.”,"[2484, 2642)"


{
  "document_title": "VINGROUP AND LEGENDS GLOBAL ANNOUNCE STRATEGIC PARTNERSHIP FOR VINFAST - TRONG DONG STADIUM",
  "section_subtitle": "Vingroup and Legends Global announced a strategic partnership to manage and operate VinFast – Trong Dong Stadium\\.",
  "section_content": "\nHanoi, Vietnam, Vingroup - Vietnam’s largest private conglomerate, and Legends Global, the premier partner to the world’s greatest live events, venues and brands, have announced a strategic partnership for the management and operation of VinFast – Trong Dong Stadium. The agreement marks a significant milestone ahead of the landmark venue’s scheduled opening on July 31, 2027.\n\nUnder the agreement, Vingroup and Legends Global will work together to establish VinFast – Trong Dong Stadium as a premier global destination for world-class sporting, cultural, and entertainment events. In addition to helping attract major international events to the stadium, Legends Global will collaborate with Vingroup to oversee ve

## Run live extraction

Run the next cell to call Jev and Luna. Provider errors surface with the failing section, sentence, and stage; they are not treated as “no claims.” Failed semantic checks receive one repair attempt. Unresolved ambiguity rejects the sentence. No context is borrowed from neighboring sections.

In [14]:
missing_keys = [
    key for key in ("TYPESAFE_API_KEY", "OPENAI_API_KEY")
    if not os.getenv(key, "").strip()
]
if missing_keys:
    raise RuntimeError("Set these environment variables before extraction: " + ", ".join(missing_keys))

result = None  # Avoid displaying a stale result if a rerun fails.
started = perf_counter()
result = await extract_claims(
    markdown_doc,
    question=question,
    decision_model=decision_model,
    generation_model=generation_model,
    max_concurrency=max_concurrency,
)
elapsed_seconds = perf_counter() - started
print(f"Extracted {len(result.claims)} claims in {elapsed_seconds:.2f}s")
print(dict(Counter(sentence.status for sentence in result.sentences)))

Extracted 35 claims in 64.29s
{'validation_failed': 20, 'no_verifiable_claims': 2, 'extracted': 11}


## Review claims and sentence outcomes

Check the source alongside each claim, particularly attribution, quantities, uncertainty, and future plans. `validation_failed` means repair exhausted; `unresolved_ambiguity` means the section did not support a clear interpretation.

In [15]:
claim_rows = [
    {
        "subtitle": result.document.sections[claim.section_index].subtitle,
        "sentence": claim.sentence_index,
        "claim": claim.text,
        "source": markdown_doc[claim.start:claim.end],
    }
    for claim in result.claims
]
show_table(claim_rows, [("Subtitle", "subtitle"), ("Sentence", "sentence"), ("Claim", "claim"), ("Original source", "source")])

show_table(
    [sentence.model_dump() for sentence in result.sentences],
    [("Sentence", "index"), ("Status", "status"), ("Original", "text"),
     ("Filtered", "selected_text"), ("Clarified", "clarified_text")],
)

Subtitle,Sentence,Claim,Original source
"Mr\. Peter Loxton\, Chief Operating Officer for Asia Pacific and the Middle East at Legends Global\, said the stadium’s first major event would deliver an experience matching its extraordinary scale\.",11,Peter Loxton said that the long-term exclusive partnership with Vingroup allows Legends Global to combine global expertise with local talent to strengthen Vietnam’s venue management capabilities.,Our long-term exclusive partnership with Vingroup allows Legends Global to combine global expertise with local talent to strengthen Vietnam’s venue management capabilities.”
"VinFast – Trong Dong Stadium features a distinctive\, one\-of\-a\-kind design",15,An extensive ecosystem of training grounds and large-scale sports facilities surrounds the stadium within the Hung Vuong International Sports Complex.,Surrounding the stadium is an extensive ecosystem of training grounds and large-scale sports facilities within the Hung Vuong International Sports Complex.
"VinFast – Trong Dong Stadium features a distinctive\, one\-of\-a\-kind design",19,The partnership [between Vingroup and Legends Global] lays the foundation for a long-term collaboration between the two organizations.,"The partnership also lays the foundation for a long-term collaboration between the two organizations, with Legends Global providing exclusive stadium, arena, and live event services to Vingroup in Vietnam."
"VinFast – Trong Dong Stadium features a distinctive\, one\-of\-a\-kind design",19,"The collaboration [between Vingroup and Legends Global] includes Legends Global providing exclusive stadium, arena, and live event services to Vingroup in Vietnam.","The partnership also lays the foundation for a long-term collaboration between the two organizations, with Legends Global providing exclusive stadium, arena, and live event services to Vingroup in Vietnam."
"VinFast – Trong Dong Stadium features a distinctive\, one\-of\-a\-kind design",21,Recruitment for the stadium’s operations team is actively underway.,"Meanwhile, recruitment for the stadiums operations team is actively underway, establishing the groundwork for the venue to commence operations in accordance with international standards./."
"VinFast – Trong Dong Stadium features a distinctive\, one\-of\-a\-kind design",21,The recruitment is establishing the groundwork for the venue to commence operations in accordance with international standards.,"Meanwhile, recruitment for the stadiums operations team is actively underway, establishing the groundwork for the venue to commence operations in accordance with international standards./."
ABOUT VINGROUP,22,Vingroup is Vietnam’s largest private conglomerate.,Vingroup is Vietnams largest private conglomerate and was ranked 57th on TIME magazines list of the Best Companies in Asia-Pacific 2026.
ABOUT VINGROUP,22,Vingroup was ranked 57th on TIME magazine’s list of the Best Companies in Asia-Pacific 2026.,Vingroup is Vietnams largest private conglomerate and was ranked 57th on TIME magazines list of the Best Companies in Asia-Pacific 2026.
ABOUT VINGROUP,24,"Today, companies within the Vingroup ecosystem operate in 12 countries.","Today, companies within the Vingroup ecosystem operate in 12 countries, providing employment for more than 400,000 people across six core pillars: Industrials & Technology, Real Estate & Services, Infrastructure, Green Energy, Social Enterprises, and Culture & Sports."
ABOUT VINGROUP,24,"Today, companies within the Vingroup ecosystem provide employment for more than 400,000 people across six core pillars: Industrials & Technology, Real Estate & Services, Infrastructure, Green Energy, Social Enterprises, and Culture & Sports.","Today, companies within the Vingroup ecosystem operate in 12 countries, providing employment for more than 400,000 people across six core pillars: Industrials & Technology, Real Estate & Services, Infrastructure, Green Energy, Social Enterprises, and Culture & Sports."


Sentence,Status,Original,Filtered,Clarified
0,validation_failed,"Hanoi, Vietnam, Vingroup - Vietnam’s largest private conglomerate, and Legends Global, the premier partner to the world’s greatest live events, venues and brands, have announced a strategic partnership for the management and operation of VinFast – Trong Dong Stadium.",None,None
1,validation_failed,"The agreement marks a significant milestone ahead of the landmark venue’s scheduled opening on July 31, 2027.",None,None
2,validation_failed,"Under the agreement, Vingroup and Legends Global will work together to establish VinFast – Trong Dong Stadium as a premier global destination for world-class sporting, cultural, and entertainment events.",None,None
3,validation_failed,"In addition to helping attract major international events to the stadium, Legends Global will collaborate with Vingroup to oversee venue and facility management, vendor coordination, and on-site service operations, ensuring that every aspect meets the highest international standards of professionalism and excellence.",None,None
4,validation_failed,"Drawing on its extensive experience managing more than 450 venues worldwide, hosting over 20,000 major events, and welcoming more than 165 million guests annually, Legends Global will also play a key role in developing the expertise and capabilities of Vingroup’s venue management professionals in Vietnam.",None,None
5,validation_failed,"Commenting on the strategic partnership, Mr. Nguyen Viet Quang, Vice Chairman and CEO of Vingroup, said: “VinFast - Trong Dong Stadium is far more than a sporting venue.",None,None
6,validation_failed,"With its world-leading scale, contemporary architecture inspired by Vietnamese cultural heritage, automated retractable roof, and playing surface designed to meet FIFA standards, we envision working alongside Legends Global to transform the stadium into a global gathering place for the finest sporting, cultural, and entertainment events.”",None,None
7,validation_failed,"Mr. Dan Levy, CEO of Legends Global, commented: “VinFast – Trong Dong Stadium is an extraordinary project for Vietnam, one that will help position Hanoi at the forefront of the global live events industry.",None,None
8,validation_failed,Legends Global is honored to partner with Vingroup on this landmark development.,None,None
9,no_verifiable_claims,We look forward to sharing our expertise in venue and event management to help bring Vingroup’s ambition and vision for the world’s largest stadium to life.”,None,None


## Inspect decisions, confidence, and usage

Choose a sentence index to inspect every stage, including repair attempts. Confidence metadata comes from the provider and is not a guarantee of correctness. Resolved model identifiers and per-stage usage are retained in the export.

In [ ]:
sentence_index = 0
if result.sentences:
    sentence = next(s for s in result.sentences if s.index == sentence_index)
    for stage in sentence.stages:
        print(json.dumps(stage.model_dump(mode="json"), ensure_ascii=False, indent=2))
else:
    print("No sentences to inspect.")

print("Requested models:", result.requested_models)
print("Resolved models:", sorted({stage.model for s in result.sentences for stage in s.stages if stage.model}))
usage_totals = Counter()
for sentence in result.sentences:
    for stage in sentence.stages:
        for key, value in stage.usage.items():
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                usage_totals[key] += value
print("Numeric usage totals:", dict(usage_totals))

## Optional JSON export

Set `save_results=True` to write the full result, original Markdown, and runtime metadata beneath `.data/claimify/`. Each export uses a unique filename. The original input is included so claim offsets remain interpretable.

In [ ]:
save_results = False
if save_results:
    from datetime import datetime, timezone
    from uuid import uuid4

    output_dir = project_root / ".data" / "claimify"
    output_dir.mkdir(parents=True, exist_ok=True)
    stem = input_path.stem
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    output_path = output_dir / f"{stem}-{timestamp}-{uuid4().hex[:8]}.json"
    payload = {
        "source_markdown": markdown_doc,
        "question": question,
        "elapsed_seconds": elapsed_seconds,
        "result": result.model_dump(mode="json"),
    }
    with output_path.open("x", encoding="utf-8") as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2)
        stream.write("\n")
    print(output_path)
else:
    print("Export disabled; set save_results=True to save.")